In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

import cv2
import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl

from box import Box
import yaml

import matplotlib.pyplot as plt

import random 
import time 

root_dir = os.path.abspath('..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.models.dpso_train import DPSO_train as DPSO
from src.train.lightning_module import DPSO_LightningModule
from src.data_loader.data_module_lightning import SonarSimDataModule
from src.data_loader.transforms import SonarDatasetTranforms
from training.metrics import pose_err


In [ ]:
# model configuration
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'

# input data
root_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data'

# output data 
output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/training/test_output2'


# training configuration
epochs = 10
batch_size = 1
num_workers = 0
transforms = SonarDatasetTranforms

frames_in_series = 6
init_frames = 3
freeze_poses_step = 4
init_poses_noise = 1.0
loss_weights = []
opt_iter_weights = []

# ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(model_config_pth, "r") as f:
            model_config = Box(yaml.safe_load(f))



In [ ]:
dpso = DPSO(model_config_pth, sonar_config_pth, batch_size, frames_in_series, init_frames, device)
model = DPSO_LightningModule(dpso, freeze_poses_step, init_poses_noise, loss_weights, opt_iter_weights)
dm = SonarSimDataModule(root_dir, batch_size, num_workers, transforms, frames_in_series)
dm.setup()


In [ ]:
trainer = pl.Trainer(
    max_epochs=epochs,               # Ile epok trenować
    accelerator="auto",          # Automatycznie wykryje GPU, MPS (Mac) lub CPU
    devices="auto",              # Użyje wszystkich dostępnych rdzeni/kart
    log_every_n_steps=10         # Jak często odświeżać logi
    # precision="16-mixed"         # Opcjonalnie: przyspiesza trening na nowszych GPU
)

In [ ]:
trainer.fit(model, datamodule=dm)